---
title: "Characteristic-Wise WENO-Z for Euler Shocks"
date: 2026-10-14
categories: [high-order methods, WENO, compressible flow]
description: "Compare component-wise and Roe-characteristic WENO-Z reconstruction on fixed-grid Euler tests."
---

# Characteristic-Wise WENO-Z for Euler Shocks
### Same nonlinear stencil logic, different reconstruction space

**Learning goal:** Implement fifth-order WENO-Z state reconstruction in both conservative-component and Roe-characteristic variables, then quantify how reconstruction space changes oscillation control, contact/entropy-wave resolution, conservation, and robustness.

**Time:** 60-90 minutes. Python, NumPy, Matplotlib. Use the prescribed fixed grids only; this is not a grid-refinement or order-estimation exercise.

You already know the basic WENO machinery. The focus here is the production-solver decision: where the nonlinear weights are computed, how the characteristic basis is frozen at an interface, and what diagnostics distinguish a clean-looking answer from a trustworthy one.

All implementation cells are intentionally TODOs. No completed solver is supplied.

## 1. Governing equations and finite-volume update

Solve the one-dimensional Euler equations
$$U_t+F(U)_x=0,\qquad U=[\rho,\rho u,E]^T,$$
$$F(U)=[\rho u,\rho u^2+p,u(E+p)]^T,\qquad p=(\gamma-1)\left(E-\tfrac12\rho u^2\right),$$
with $\gamma=1.4$. Advance cell averages using
$$\frac{dU_i}{dt}=-\frac{\hat F_{i+1/2}-\hat F_{i-1/2}}{\Delta x}$$
and SSPRK(3,3). Set each time step from the maximum local signal speed $|u|+c$ and CFL = 0.35. Use three transmissive ghost cells on each side.

At an interface, reconstruct left/right conservative states and evaluate a local Rusanov flux. Keep flux evaluation identical between methods so reconstruction space is the only controlled difference.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

GAMMA = 1.4
CFL = 0.35
RHO_FLOOR = 1.0e-12
P_FLOOR = 1.0e-12
EPS_WENO = 1.0e-40

# TODO: Set deterministic print/plot options and record package versions.


## 2. WENO-Z scalar reconstruction

Implement the standard three third-order candidate polynomials and Jiang-Shu smoothness indicators $\beta_0,\beta_1,\beta_2$ for a fifth-order left-biased interface value. Use optimal weights $d=(0.1,0.6,0.3)$ and WENO-Z weights
$$\tau_5=|\beta_0-\beta_2|,\qquad
\alpha_k=d_k\left[1+\left(\frac{\tau_5}{\beta_k+\epsilon}\right)^2\right],\qquad
\omega_k=\frac{\alpha_k}{\sum_j\alpha_j}.$$

Obtain the right-biased reconstruction by symmetry, not a second unrelated formula. Return both the reconstructed value and weights so the nonlinear stencil choice can be inspected. Unit-test constant, linear, and mirrored stencil behavior before using Euler states.

In [ ]:
def weno_z_left(v5, eps=EPS_WENO):
    """Return left-biased interface value and three nonlinear weights."""
    # TODO: Implement WENO-Z from a length-5 scalar stencil.
    raise NotImplementedError

def weno_z_right(v5, eps=EPS_WENO):
    # TODO: Implement by a verified symmetry transformation.
    raise NotImplementedError

# TODO: Add scalar reconstruction unit tests before continuing.


## 3. Two reconstruction spaces

**Component-wise:** Apply scalar WENO-Z independently to the three conservative components on each interface stencil.

**Characteristic-wise:** At every interface, compute a Roe-averaged state from the adjacent cell averages, construct right and left eigenvector matrices $R$ and $L=R^{-1}$ for the Euler flux Jacobian, and freeze this basis across the full five-point stencil. Project each stencil vector with $w_j=LU_j$, reconstruct each characteristic component, then return with $U^\pm=Rw^\pm$.

Verify numerically that $LR\approx I$, eigenvalues are $\tilde u-\tilde c,\tilde u,\tilde u+\tilde c$, and both methods exactly preserve a constant state. Do not recompute a different eigenbasis at every stencil point; that destroys the intended local characteristic projection.

In [ ]:
def primitive_to_conservative(rho, u, p):
    # TODO
    raise NotImplementedError

def conservative_to_primitive(U):
    # TODO: Return rho, u, p, c and reject nonphysical states clearly.
    raise NotImplementedError

def roe_eigenvectors(UL, UR):
    # TODO: Return eigenvalues, R, and L at one interface.
    raise NotImplementedError

def reconstruct_interfaces(Ug, mode):
    """mode is 'component' or 'characteristic'; return UL, UR and weight diagnostics."""
    # TODO: Reconstruct every physical interface from ghost-padded cell averages.
    raise NotImplementedError


## 4. Flux, time integration, and admissibility fallback

Use the local Rusanov flux
$$\hat F(U^-,U^+)=\tfrac12[F(U^-)+F(U^+)]-\tfrac12a_{max}(U^+-U^-),$$
where $a_{max}=\max(|u^-|+c^-,|u^+|+c^+)$. Advance with SSPRK(3,3).

At each RK stage, first attempt the WENO-Z spatial operator. If any candidate stage has nonfinite values, $\rho\le\rho_{floor}$, or $p\le p_{floor}$, repeat that entire stage using piecewise-constant states with the same Rusanov flux. Count and report every fallback. This coarse stage-level mechanism is intentionally transparent; it is a safety net, not a claim of high-order positivity preservation.

Track total mass, momentum, and energy after every accepted full step. With transmissive boundaries, interpret conservation changes using net boundary flux, not an unjustified zero-drift assertion.

In [ ]:
def euler_flux(U):
    # TODO
    raise NotImplementedError

def rusanov_flux(UL, UR):
    # TODO
    raise NotImplementedError

def spatial_operator(U, dx, mode, first_order=False):
    # TODO: Apply ghost cells, reconstruct or use piecewise constants, and difference fluxes.
    raise NotImplementedError

def ssprk3_step(U, dt, dx, mode, counters):
    # TODO: Three stages with explicit admissibility checks and fallback accounting.
    raise NotImplementedError

def integrate(U0, x, t_end, mode):
    # TODO: CFL time loop, histories, and final state.
    raise NotImplementedError


## 5. Fixed-grid experiments

Run both reconstruction modes on exactly these cases:

**Lax shock tube:** $x\in[0,1]$, N = 400, discontinuity at 0.5, final time 0.14.
Left $(\rho,u,p)=(0.445,0.698,3.528)$; right $(0.5,0,0.571)$.

**Shu-Osher shock-entropy interaction:** $x\in[-5,5]$, N = 800, final time 1.8.
For $x<-4$, $(\rho,u,p)=(3.857143,2.629369,10.33333)$; otherwise $(1+0.2\sin(5x),0,1)$.

Use identical CFL logic and stopping at exact final time. Record steps, wall time, reconstruction calls, fallback count, minimum density/pressure, and conservation closure including accumulated boundary flux.

In [ ]:
def lax_initial_condition(n=400):
    # TODO
    raise NotImplementedError

def shu_osher_initial_condition(n=800):
    # TODO
    raise NotImplementedError

# TODO: Run both reconstruction modes on both prescribed cases and collect metrics.


## 6. Required diagnostics

Create:

1. Lax density, velocity, and pressure for both methods, with a zoom around the contact and shock.
2. Shu-Osher density over the full domain and a zoom of the post-shock entropy waves.
3. Pressure-oscillation diagnostic: plot adjacent-cell pressure jumps and report their maximum in user-defined windows around the Lax contact and shock.
4. WENO nonlinear weights at selected interfaces in a smooth entropy-wave region and immediately upstream of the shock.
5. A table containing minimum $\rho$, minimum $p$, conservation closure, fallbacks, steps, and wall time.

Quantify rather than merely claim that characteristic projection is cleaner. Report the exact windows used for jump metrics, and do not interpret lower variation everywhere as automatically better: excessive damping can also suppress oscillations.

In [ ]:
# TODO: Produce the required plots, table, and quantitative comparison.


## 7. Verification and success criteria

Assert:

- Scalar WENO-Z exactly reconstructs constants to machine precision; mirrored left/right tests pass.
- $\|LR-I\|_\infty<10^{-12}$ for at least 20 seeded physical Roe-state pairs.
- Constant Euler states remain constant for both reconstruction modes after 20 steps to below $10^{-12}$ relative error.
- Every accepted stage has finite $\rho,p$ above the floors.
- Final times are reached exactly to roundoff and both modes use the same time-step rule.
- Mass, momentum, and energy changes agree with accumulated numerical boundary flux to below $10^{-10}$ relative scale.
- Both methods use the same Rusanov flux and SSPRK3 implementation; only reconstruction space changes.
- All fallback events are reported. Zero fallback is acceptable; an unreported fallback is not.

Do not add a mesh sweep. These checks target algebra, free-stream preservation, conservation, and physical admissibility on fixed grids.

In [ ]:
# TODO: Implement automatic assertions tied to the computed histories.


## 8. Interpretation, stretch goal, and references

Answer briefly:

1. Why can component-wise reconstruction mix waves that travel at different characteristic speeds?
2. Why is one interface-local Roe basis frozen across the reconstruction stencil?
3. Where does characteristic reconstruction materially reduce pressure ringing, and where is it merely more expensive?
4. How do WENO-Z weights distinguish a smooth entropy wave from a discontinuity?
5. Why is a stage-level first-order fallback robust but overly dissipative?

**Stretch goal (15-25 minutes):** Replace the stage-level fallback with local flux fallback only at interfaces adjacent to nonphysical candidate cells. Recheck conservation closure. Alternatively, add WENO-JS weights while keeping characteristic projection fixed and compare critical-point behavior without changing the grid.

### References

- **Core implementation reference:** Chi-Wang Shu, *Essentially Non-Oscillatory and Weighted Essentially Non-Oscillatory Schemes for Hyperbolic Conservation Laws*, NASA/CR-97-206253, ICASE Report 97-65, 1997. [NASA report](https://ntrs.nasa.gov/search.jsp?R=19980007543).
- Rafael Borges, Monique Carmona, Bruno Costa, and Wai-Sun Don, "An Improved Weighted Essentially Non-Oscillatory Scheme for Hyperbolic Conservation Laws," *Journal of Computational Physics* 227(6), 3191-3211, 2008. [DOI](https://doi.org/10.1016/j.jcp.2007.11.038).
- **NASA Glenn robustness reference:** H. T. Huynh, "Shock Capturing via Limiting for High-Order Methods including Discontinuous Galerkin," 2024. [NASA Glenn NTRS record](https://ntrs.nasa.gov/citations/20240006106).

### Topic ledger

| Date | Primary objective |
|---|---|
| 2026-10-06 | Fixed-mesh advection: CFL, conservation, positivity, TV |
| 2026-10-07 | Krylov methods and line-block preconditioning |
| 2026-10-08 | Data layout, cache behavior, flux-kernel throughput |
| 2026-10-09 | Nonlinear natural-circulation thermal-fluid modeling |
| 2026-10-12 | Discrete-adjoint PDE-constrained optimization |
| 2026-10-13 | Partitioned multiphysics coupling and relaxation |
| 2026-10-14 | Characteristic high-order shock-capturing reconstruction |

All seven immediately preceding/current dated objectives are distinct. This is assignment 6 of the ten-assignment grid-study exclusion; four excluded assignments remain after today. Planned next objectives: parallel algorithms, time integration, uncertainty propagation, and turbulence modeling.